In [ ]:
# 创建客户端

In [4]:
from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")
# 列出所有库
existing_dbs = client.list_databases()
print("== Databases ==")
for db in existing_dbs:
    print(db)

== Databases ==
default
rag_tu


In [3]:
# 创建数据库
db_name = "rag_tu"
if db_name not in existing_dbs:
    client.create_database(db_name=db_name)


In [ ]:
# 删除数据库
client.drop_database(db_name=db_name)

In [9]:
# 切换数据库
client.use_database(db_name=db_name)

In [11]:
# 查看database下面的collection
collections = client.list_collections();
for collection in collections:
    print(collection)

docs


In [10]:
# 创建collection
collection_name = "docs"
client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

In [12]:
# 查看collection元数据
from rich import print as rprint

metedata = client.describe_collection(
    collection_name=collection_name
)
rprint(metedata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 467509462201286072,
    'consistency_level': 2,
    'consistency_level_name': 'Bounded',
    'properties': {'timezone': 'UTC'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 467510462392565777,
    'update_timestamp': 467510462392565777
}

In [13]:
# 向量存储
from langchain_community.document_loaders import CSVLoader
from langchain.embeddings import init_embeddings
import os
from dotenv import load_dotenv
from openai import api_key, base_url

load_dotenv(override=True)
# 初始化嵌入模型
embedding_model = init_embeddings(
    model="openai:Pro/BAAI/bge-m3",
    api_key=os.getenv("SILICONFLOW_API_KEY"),
    base_url=os.getenv("SILICONFLOW_BASE_URL")
)
loader = CSVLoader("../asset/load/02-load.csv", encoding="utf-8")
docs = loader.load_and_split()

# 存放每一个chunk的embedding
texts = [doc.page_content for doc in docs]
embeded_docs = embedding_model.embed_documents(texts)

print(len(embeded_docs))
for i in range(len(texts)):
    print(f"{texts[i]}:\n{embeded_docs[i][:3]}", end="\n\n")


4
id: 1
title: Introduction to Python
content: Python is a popular programming language.
author: John Doe:
[-0.003082296811044216, 0.013214709237217903, 0.040597766637802124]

id: 2
title: Data Science Basics
content: Data science involves statistics and machine learning.
author: Jane Smith:
[0.005167097318917513, 0.014744328334927559, 0.04923552647233009]

id: 3
title: Web Development
content: HTML, CSS and JavaScript are core web technologies.
author: Mike Johnson:
[-0.020887864753603935, 0.0333939753472805, 0.06412708014249802]

id: 4
title: Artificial Intelligence
content: AI is transforming many industries.
author: Sarah Williams:
[-0.02705685980618, 0.017125584185123444, -0.008953787386417389]

